# PMC Scraper Engine

Enhanced web scraper for PubMed Central (PMC) that automatically collects and downloads research papers with metadata. Uses Selenium for dynamic content handling and implements robust error handling, rate limiting, and multiple PDF extraction strategies for reliable paper collection from PMC search

In [ ]:
"""
Enhanced PMC scraper with better error handling, rate limiting, and debugging.
Fixes download failures by adding proper headers, retry logic, and session management.
"""
import os
import time
import random
import json
import argparse
import re
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ---------------
# Configuration
# ---------------
DEFAULT_SEARCH_URL = "https://www.ncbi.nlm.nih.gov/pmc/?term=(Pakistan%5BTitle%5D)+AND+mental+health+AND+mental+illness"
DEFAULT_OUTPUT_DIR = "D:\\PsyWiz\\raw_data"
DEFAULT_MAX_PAPERS = 3
DEFAULT_HEADLESS = True
DEFAULT_MIN_PAUSE = 1.5
DEFAULT_MAX_PAUSE = 3.0
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
PDF_SIG = b"%PDF"

def create_driver(headless=True):

    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")
    opts.add_argument(f"--user-agent={USER_AGENT}")
    opts.add_argument("--window-size=1920,1200")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option('useAutomationExtension', False)
    
    service = ChromeService(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=opts)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

def collect_article_links_and_titles(driver, start_url, max_links=100, pause_range=(1.5, 3.0)):

    results = []
    driver.get(start_url)
    time.sleep(random.uniform(*pause_range))

    while len(results) < max_links:
        try:
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.rprt")))
        except Exception:
            print("    - No more results found or timeout")
            break

        blocks = driver.find_elements(By.CSS_SELECTOR, "div.rprt")
        for b in blocks:
            if len(results) >= max_links:
                break
            try:
                a = b.find_element(By.CSS_SELECTOR, "div.title a.view")
                article_url = a.get_attribute("href")
                title = a.text.strip()
            except Exception:
                continue
            if article_url:
                article_url = urljoin(driver.current_url, article_url)
                results.append({"article": article_url, "title": title})
        
        if len(results) >= max_links:
            break

        try:
            next_buttons = driver.find_elements(By.CSS_SELECTOR, 
                "a.page_link.next, a.active.page_link.next, a.ncbi-pagination__next, a[title^='Next page']")
            next_btn = next_buttons[0] if next_buttons else None
            
            if not next_btn:
                try:
                    next_btn = driver.find_element(By.ID, "EntrezSystem2.PEntrez.PMC.Pmc_ResultsPanel.Entrez_Pager.Page")
                except Exception:
                    next_btn = None
                    
            if next_btn and next_btn.is_displayed() and next_btn.is_enabled():
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_btn)
                time.sleep(0.2)
                next_btn.click()
                time.sleep(random.uniform(*pause_range))
            else:
                break
        except Exception:
            break

    return results[:max_links]

def transfer_selenium_cookies_to_session(driver, session, domain=None):

    for c in driver.get_cookies():
        if domain and domain not in c.get("domain", ""):
            continue
        session.cookies.set(c["name"], c["value"], domain=c.get("domain"), path=c.get("path", "/"))
    return session

def find_pdf_url_on_article(driver, article_url, session, pause=1.0):

    try:
        print(f"    - Navigating to article page...")
        driver.get(article_url)
        time.sleep(pause)
        
        transfer_selenium_cookies_to_session(driver, session, domain="ncbi.nlm.nih.gov")
        
    except Exception as e:
        print(f"    - Failed to load article page: {e}")
        return None

    try:
        selectors = [
            "a[data-ga-label*='pdf']",
            "a[href*='/pdf/']", 
            "a[href$='.pdf']",
            "a[title*='PDF']",
            "a[title*='pdf']",
            "a[data-track-action*='pdf']"
        ]
        
        for selector in selectors:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)
            for elem in elems:
                href = elem.get_attribute("href")
                if href and "javascript" not in href.lower() and href != "#":
                    abs_url = urljoin(article_url, href)
                    print(f"    - Found PDF link via selector {selector}: {abs_url}")
                    return abs_url
    except Exception:
        pass

    try:
        current_url = driver.current_url
        if "/articles/PMC" in current_url:
            pmc_match = re.search(r'/articles/(PMC\d+)', current_url)
            if pmc_match:
                pmc_id = pmc_match.group(1)
                possible_patterns = [
                    f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmc_id}/pdf/",
                    f"https://pmc.ncbi.nlm.nih.gov/articles/{pmc_id}/pdf/",
                ]
                
                for pattern in possible_patterns:
                    try:
                        test_resp = session.head(pattern, timeout=10, allow_redirects=True)
                        if test_resp.status_code == 200:
                            ct = test_resp.headers.get("content-type", "").lower()
                            if "pdf" in ct:
                                print(f"    - Found PDF via pattern: {pattern}")
                                return pattern
                    except Exception:
                        continue
    except Exception:
        pass

    try:
        page_source = driver.page_source
        soup = BeautifulSoup(page_source, "html.parser")
        meta = soup.find("meta", attrs={"name": "citation_pdf_url"})
        if meta and meta.get("content"):
            pdf_url = urljoin(article_url, meta["content"])
            print(f"    - Found PDF via meta tag: {pdf_url}")
            return pdf_url
    except Exception:
        pass

    try:
        button_selectors = [
            "//a[contains(text(),'PDF') or contains(text(),'pdf')]",
            "//button[contains(text(),'PDF')]",
            "//a[contains(text(),'Download')]",
            "//a[@class='int-view']",
            "//a[contains(@class, 'pdf')]"
        ]
        
        for selector in button_selectors:
            buttons = driver.find_elements(By.XPATH, selector)
            for btn in buttons[:2]: 
                try:
                    initial_url = driver.current_url
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                    time.sleep(0.2)
                    btn.click()
                    time.sleep(pause)
                    
                    new_url = driver.current_url
                    if new_url != initial_url and ("/pdf/" in new_url or new_url.endswith(".pdf")):
                        print(f"    - Found PDF via button click: {new_url}")
                        return new_url
                    
                    # Check for iframes
                    frames = driver.find_elements(By.TAG_NAME, "iframe")
                    for frame in frames:
                        src = frame.get_attribute("src")
                        if src and ("/pdf/" in src or src.endswith(".pdf")):
                            abs_src = urljoin(article_url, src)
                            print(f"    - Found PDF via iframe: {abs_src}")
                            return abs_src
                            
                except Exception:
                    continue
    except Exception:
        pass

    try:
        headers = {
            "User-Agent": USER_AGENT,
            "Referer": "https://www.ncbi.nlm.nih.gov/",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
        }
        resp = session.get(article_url, headers=headers, timeout=30)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "html.parser")
            
            for a in soup.find_all("a", href=True):
                href = a["href"]
                if "/pdf/" in href or href.endswith(".pdf") or "format=pdf" in href:
                    abs_href = urljoin(article_url, href)
                    print(f"    - Found PDF via HTML parsing: {abs_href}")
                    return abs_href
    except Exception:
        pass

    print("    - No PDF URL found with any method")
    return None

def find_and_download_pdf_from_viewer(driver, pdf_viewer_url, out_path, session, timeout=90):
    
    try:
        print(f"    - Loading PDF viewer page: {pdf_viewer_url}")
        driver.get(pdf_viewer_url)
        time.sleep(3)  
        
        download_selectors = [
            "cr-icon-button#download",
            "cr-icon-button[iron-icon='cr:file-download']",
            "cr-icon-button[aria-label='Download']",
            "cr-icon-button[title='Download']",
            "#download",
            "[iron-icon='cr:file-download']",
            "[aria-label='Download']",
            "[title='Download']",
            "viewer-download-controls cr-icon-button",
            "viewer-toolbar cr-icon-button#download"
        ]
        
        xpath_selectors = [
            "/html/body/pdf-viewer//viewer-toolbar//div/div[3]/viewer-download-controls//cr-icon-button",
            "//cr-icon-button[@id='download']",
            "//cr-icon-button[@iron-icon='cr:file-download']",
            "//cr-icon-button[@aria-label='Download']",
            "//viewer-download-controls//cr-icon-button",
            "//pdf-viewer//cr-icon-button[@id='download']"
        ]
        
        download_btn = None
        
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.TAG_NAME, "pdf-viewer"))
            )
            print("    - PDF viewer component loaded")
        except Exception:
            print("    - PDF viewer component not found, continuing anyway...")
        
        for selector in download_selectors:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                if elements:
                    for elem in elements:
                        if elem.is_displayed() and elem.is_enabled():
                            download_btn = elem
                            print(f"    - Found download button via CSS selector: {selector}")
                            break
                    if download_btn:
                        break
            except Exception as e:
                print(f"    - CSS selector {selector} failed: {e}")
                continue
        
        if not download_btn:
            for xpath in xpath_selectors:
                try:
                    elements = driver.find_elements(By.XPATH, xpath)
                    if elements:
                        for elem in elements:
                            if elem.is_displayed() and elem.is_enabled():
                                download_btn = elem
                                print(f"    - Found download button via XPath: {xpath}")
                                break
                        if download_btn:
                            break
                except Exception as e:
                    print(f"    - XPath {xpath} failed: {e}")
                    continue
        
        if not download_btn:
            print("    - Download button not found with standard selectors, trying aggressive search...")
            time.sleep(2)  
            
            try:

                all_elements = driver.find_elements(By.XPATH, "//*[contains(@aria-label, 'Download') or contains(@title, 'Download') or @id='download']")
                for elem in all_elements:
                    try:
                        if elem.is_displayed() and elem.is_enabled():
                            tag_name = elem.tag_name
                            element_id = elem.get_attribute("id")
                            aria_label = elem.get_attribute("aria-label")
                            title = elem.get_attribute("title")
                            print(f"    - Found potential download element: {tag_name}, id={element_id}, aria-label={aria_label}, title={title}")
                            
                            if any(attr and "download" in attr.lower() for attr in [element_id, aria_label, title]):
                                download_btn = elem
                                print("    - Using this element as download button")
                                break
                    except Exception:
                        continue
                        
            except Exception:
                pass
        
        if not download_btn:
            print(f"    - Download button not found")

            return try_direct_pdf_download(pdf_viewer_url, out_path, session, timeout)

        try:
            success, result = try_direct_pdf_download(pdf_viewer_url, out_path, session, timeout)
            if success:
                return True, result
        except Exception as e:
            print(f"    - Direct URL method failed: {e}")

        if download_btn:
            try:
                print("    - Attempting to click download button...")
                
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", download_btn)
                time.sleep(0.5)
                
                click_successful = False
                
                # Method 1: Regular click
                try:
                    download_btn.click()
                    click_successful = True
                    print("    - Clicked download button (normal click)")
                except Exception as e:
                    print(f"    - Normal click failed: {e}")
                
                # Method 2: JavaScript click
                if not click_successful:
                    try:
                        driver.execute_script("arguments[0].click();", download_btn)
                        click_successful = True
                        print("    - Clicked download button (JavaScript click)")
                    except Exception as e:
                        print(f"    - JavaScript click failed: {e}")
                
                # Method 3: Dispatch click event
                if not click_successful:
                    try:
                        driver.execute_script("""
                            var event = new MouseEvent('click', {
                                view: window,
                                bubbles: true,
                                cancelable: true
                            });
                            arguments[0].dispatchEvent(event);
                        """, download_btn)
                        click_successful = True
                        print("    - Clicked download button (event dispatch)")
                    except Exception as e:
                        print(f"    - Event dispatch failed: {e}")
                
                if click_successful:

                    download_dir = os.path.dirname(out_path)
                    initial_files = set(os.listdir(download_dir)) if os.path.exists(download_dir) else set()
                    
                    for i in range(30):
                        time.sleep(1)
                        try:
                            current_files = set(os.listdir(download_dir))
                            new_files = current_files - initial_files
                            pdf_files = [f for f in new_files if f.endswith('.pdf')]
                            
                            if pdf_files:

                                downloaded_file = os.path.join(download_dir, pdf_files[0])
                                if os.path.exists(downloaded_file):
                                    if downloaded_file != out_path:
                                        os.rename(downloaded_file, out_path)
                                    print(f"    - Successfully downloaded via button click: {out_path}")
                                    return True, out_path
                        except Exception:
                            continue
                    
                    print("    - Download via button click timed out or no file appeared")
                else:
                    print("    - All click methods failed")
                
            except Exception as e:
                print(f"    - Button click process failed: {e}")
        
        print("    - PDF viewer download failed")
        return False, "viewer_download_failed"
        
    except Exception as e:
        print(f"    - PDF viewer navigation error: {repr(e)}")
        return False, "viewer_error"

def download_pdf_validated(pdf_url, out_path, session, driver=None, timeout=90):

    headers = {
        "User-Agent": USER_AGENT,
        "Referer": "https://www.ncbi.nlm.nih.gov/",
        "Accept": "application/pdf,application/octet-stream,*/*",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1"
    }
    
    try:
        print(f"    - Attempting to download: {pdf_url}")
        
        # First check if this is a PDF viewer URL
        if driver and "pmc.ncbi.nlm.nih.gov" in pdf_url and "/pdf/" in pdf_url:
            print("    - Detected PMC PDF viewer URL, using viewer download method")
            return find_and_download_pdf_from_viewer(driver, pdf_url, out_path, session, timeout)
        
        h = None
        ct = ""
        try:
            h = session.head(pdf_url, headers=headers, allow_redirects=True, timeout=20)
            print(f"    - HEAD response: {h.status_code}")
            if h.status_code == 429:
                print("    - Rate limited (429), sleeping 10 seconds...")
                time.sleep(10)
                return False, "rate_limited"
            elif h.status_code == 403:
                print("    - Access forbidden (403)")
                return False, "forbidden"

            ct = h.headers.get("content-type", "").lower() if h and hasattr(h, 'headers') else ""
            print(f"    - Content-Type from HEAD: {ct}")
        except Exception as e:
            print(f"    - HEAD request failed: {e}")
            h = None
            ct = ""

        max_retries = 3
        for attempt in range(max_retries):
            try:
                with session.get(pdf_url, headers=headers, stream=True, timeout=timeout, allow_redirects=True) as r:
                    print(f"    - GET response: {r.status_code}")
                    print(f"    - Final URL: {r.url}")
                    final_ct = r.headers.get('content-type', 'unknown')
                    print(f"    - Final Content-Type: {final_ct}")
                    
                    if r.status_code == 429:
                        print(f"    - Rate limited on attempt {attempt + 1}, waiting...")
                        time.sleep(5 * (attempt + 1))
                        continue
                    
                    r.raise_for_status()
                    
                    # Get first chunk to inspect
                    iterator = r.iter_content(chunk_size=8192)
                    first_chunk = b""
                    try:
                        first_chunk = next(iterator)
                    except StopIteration:
                        first_chunk = b""
                    
                    # Check if it's actually a PDF
                    is_pdf = (first_chunk.startswith(PDF_SIG) or "application/pdf" in final_ct.lower())
                    
                    if is_pdf:
                        print("    - Confirmed PDF content, downloading...")
                        total = int(r.headers.get("content-length", 0)) if r.headers.get("content-length") else 0
                        with open(out_path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=os.path.basename(out_path)) as pbar:
                            f.write(first_chunk)
                            pbar.update(len(first_chunk))
                            for chunk in iterator:
                                if chunk:
                                    f.write(chunk)
                                    pbar.update(len(chunk))
                        
                        with open(out_path, "rb") as f:
                            file_start = f.read(4)
                            if file_start.startswith(PDF_SIG[:4]):
                                return True, out_path
                            else:
                                print("    - File saved but doesn't start with PDF signature")
                                os.remove(out_path)
                                return False, "invalid_pdf"
                    else:
                        print(f"    - Response was not a PDF (Content-Type: {final_ct})")
                        return False, "not_pdf"
                
                break  
                
            except requests.exceptions.RequestException as e:
                print(f"    - Request failed on attempt {attempt + 1}: {e}")
                if attempt == max_retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
                
    except Exception as e:
        print(f"    - Download exception: {e}")
        return False, "download_error"

def try_direct_pdf_download(pdf_viewer_url, out_path, session, timeout=90):

    try:
        print(f"    - Trying direct PDF download methods for: {pdf_viewer_url}")
        
        # PMC URLs often follow patterns like:
        # Viewer: https://pmc.ncbi.nlm.nih.gov/articles/PMC7453804/pdf/S2056472420000666a.pdf
        # Direct: https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7453804/pdf/S2056472420000666a.pdf
        
        patterns_to_try = []
        
        if "pmc.ncbi.nlm.nih.gov" in pdf_viewer_url:

            direct_url = pdf_viewer_url.replace("pmc.ncbi.nlm.nih.gov", "www.ncbi.nlm.nih.gov/pmc")
            patterns_to_try.append(direct_url)
            
            patterns_to_try.extend([
                pdf_viewer_url.replace("pmc.ncbi.nlm.nih.gov", "ftp.ncbi.nlm.nih.gov/pmc"),
                pdf_viewer_url.replace("pmc.ncbi.nlm.nih.gov", "eutils.ncbi.nlm.nih.gov/pmc")
            ])
        
        base_url = pdf_viewer_url.split('?')[0] 
        patterns_to_try.extend([
            base_url + "?download=true",
            base_url + "?dl=1", 
            base_url + "?format=pdf",
            base_url + "/download"
        ])
        
        for pattern in patterns_to_try:
            try:
                print(f"    - Trying URL pattern: {pattern}")
                success, result = download_pdf_validated(pattern, out_path, session, None, timeout)
                if success:
                    return True, result
            except Exception as e:
                print(f"    - Pattern {pattern} failed: {e}")
                continue
                
        return False, "direct_download_failed"
        
    except Exception as e:
        print(f"    - Direct download method failed: {e}")
        return False, "direct_download_error"

def extract_metadata(article_url, session):
    """Extract metadata from article page"""
    headers = {
        "User-Agent": USER_AGENT,
        "Referer": "https://www.ncbi.nlm.nih.gov/"
    }
    try:
        r = session.get(article_url, headers=headers, timeout=30)
    except Exception:
        return {}
    if r.status_code != 200:
        return {}
    
    soup = BeautifulSoup(r.text, "html.parser")
    meta = {}
    
    for name in ["citation_title", "citation_author", "citation_doi", 
                 "citation_journal_title", "citation_publication_date", "citation_pmcid"]:
        tag = soup.find("meta", attrs={"name": name})
        if tag and tag.get("content"):
            meta[name] = tag["content"]
    
    if "citation_title" not in meta:
        el = soup.select_one("div.title a.view")
        if el:
            meta["title_fallback"] = el.get_text(strip=True)
    
    return meta

def sanitize_filename(name):
    """Sanitize filename for Windows"""
    keep = (" ", ".", "_", "-")
    return "".join(c for c in name if c.isalnum() or c in keep).rstrip()

def create_session():
    """Create requests session with retry strategy"""
    session = requests.Session()
    session.headers.update({
        "User-Agent": USER_AGENT,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1"
    })
    
    retry_strategy = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    
    return session

def main(args):

    SEARCH_URL = args.search_url
    OUTPUT_DIR = args.output_dir
    MAX_PAPERS = args.max_papers
    HEADLESS = args.headless
    MIN_PAUSE = args.min_pause
    MAX_PAUSE = args.max_pause

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    driver = create_driver(headless=HEADLESS)
    
    try:
        print(f"[*] Collecting up to {MAX_PAPERS} articles from PMC search results...")
        items = collect_article_links_and_titles(driver, SEARCH_URL, max_links=MAX_PAPERS, 
                                                pause_range=(MIN_PAUSE, MAX_PAUSE))
        print(f"[+] Found {len(items)} article links (cap={MAX_PAPERS}).")

        session = create_session()
        downloaded = 0
        
        for idx, itm in enumerate(items):
            if downloaded >= MAX_PAPERS:
                break
                
            article_url = itm["article"]
            title = itm.get("title", "")
            print(f"\n[{idx+1}/{len(items)}] {title}\n    {article_url}")

            parsed = urlparse(article_url)
            base = os.path.basename(parsed.path) or f"article_{idx+1}"
            base = sanitize_filename(base)
            pdf_path = os.path.join(OUTPUT_DIR, f"{base}.pdf")
            meta_path = os.path.join(OUTPUT_DIR, f"{base}_meta.json")

            transfer_selenium_cookies_to_session(driver, session, domain="ncbi.nlm.nih.gov")

            meta = extract_metadata(article_url, session)
            meta.update({"source_url": article_url, "title_from_results": title})

            pdf_url = find_pdf_url_on_article(driver, article_url, session, pause=1.0)
            if not pdf_url:
                print("    - Could not find PDF link for this article (saving metadata and skipping).")
                with open(meta_path, "w", encoding="utf-8") as mf:
                    json.dump(meta, mf, indent=2)
                time.sleep(random.uniform(MIN_PAUSE, MAX_PAUSE))
                continue

            print(f"    - PDF URL: {pdf_url}")

            if os.path.exists(pdf_path) and not args.overwrite:
                print("    - PDF exists, skipping download.")
            else:

                if "pmc.ncbi.nlm.nih.gov" in pdf_url and "/pdf/" in pdf_url:
                    ok, saved = find_and_download_pdf_from_viewer(driver, pdf_url, pdf_path, session)
                else:
                    ok, saved = download_pdf_validated(pdf_url, pdf_path, session, driver)
                    
                if not ok:
                    print(f"    - Download failed: {saved}")
                    with open(meta_path, "w", encoding="utf-8") as mf:
                        json.dump(meta, mf, indent=2)
                    time.sleep(random.uniform(MIN_PAUSE, MAX_PAUSE))
                    continue
                else:
                    print(f"    - Saved PDF: {saved}")

            try:
                with open(meta_path, "w", encoding="utf-8") as mf:
                    json.dump(meta, mf, indent=2)
                print(f"    - Wrote metadata: {meta_path}")
            except Exception as e:
                print("    - Failed writing metadata:", e)

            downloaded += 1
            time.sleep(random.uniform(MIN_PAUSE, MAX_PAUSE))

        print(f"\nDone. Downloaded & processed {downloaded} papers to: {OUTPUT_DIR}")

    finally:
        try:
            driver.quit()
        except Exception:
            pass

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Enhanced PMC scraper with better error handling")
    parser.add_argument("--search-url", type=str, default=DEFAULT_SEARCH_URL, 
                       help="PMC search URL")
    parser.add_argument("--output-dir", type=str, default=DEFAULT_OUTPUT_DIR, 
                       help="Output directory")
    parser.add_argument("--max-papers", type=int, default=DEFAULT_MAX_PAPERS, 
                       help="Maximum papers to download")
    parser.add_argument("--headless", action="store_true", default=DEFAULT_HEADLESS, 
                       help="Run browser headless")
    parser.add_argument("--min-pause", type=float, default=DEFAULT_MIN_PAUSE, 
                       help="Minimum pause between operations")
    parser.add_argument("--max-pause", type=float, default=DEFAULT_MAX_PAUSE, 
                       help="Maximum pause between operations")
    parser.add_argument("--overwrite", action="store_true", 
                       help="Overwrite existing files")

    args, _ = parser.parse_known_args()
    main(args)

[*] Collecting up to 3 articles from PMC search results...
[+] Found 3 article links (cap=3).

[1/3] Stigma toward mental and physical illness: attitudes of healthcare professionals, healthcare students and the general public in Pakistan
    https://pmc.ncbi.nlm.nih.gov/articles/PMC7453804/
    - Navigating to article page...
    - Found PDF link via selector a[data-ga-label*='pdf']: https://pmc.ncbi.nlm.nih.gov/articles/PMC7453804/pdf/S2056472420000666a.pdf
    - PDF URL: https://pmc.ncbi.nlm.nih.gov/articles/PMC7453804/pdf/S2056472420000666a.pdf
    - Loading PDF viewer page: https://pmc.ncbi.nlm.nih.gov/articles/PMC7453804/pdf/S2056472420000666a.pdf
    - PDF viewer component not found, continuing anyway...
    - Download button not found with standard selectors, trying aggressive search...
    - Download button not found
    - Trying direct PDF download methods for: https://pmc.ncbi.nlm.nih.gov/articles/PMC7453804/pdf/S2056472420000666a.pdf
    - Trying URL pattern: https://www.ncb

article_2.pdf: 100%|██████████| 841k/841k [00:00<00:00, 922kB/s] 


    - Saved PDF: D:\PsyWiz\raw_data\article_2.pdf
    - Wrote metadata: D:\PsyWiz\raw_data\article_2_meta.json

[3/3] Impact of household food insecurity and nutrition on depression and anxiety symptoms among adolescents living in rural Pakistan
    https://pmc.ncbi.nlm.nih.gov/articles/PMC12322787/
    - Navigating to article page...
    - Found PDF link via selector a[data-ga-label*='pdf']: https://pmc.ncbi.nlm.nih.gov/articles/PMC12322787/pdf/S205442512510006Xa.pdf
    - PDF URL: https://pmc.ncbi.nlm.nih.gov/articles/PMC12322787/pdf/S205442512510006Xa.pdf
    - Loading PDF viewer page: https://pmc.ncbi.nlm.nih.gov/articles/PMC12322787/pdf/S205442512510006Xa.pdf
    - PDF viewer component not found, continuing anyway...
    - Download button not found with standard selectors, trying aggressive search...
    - Download button not found
    - Trying direct PDF download methods for: https://pmc.ncbi.nlm.nih.gov/articles/PMC12322787/pdf/S205442512510006Xa.pdf
    - Trying URL pattern: ht

article_3.pdf: 100%|██████████| 452k/452k [00:00<00:00, 920kB/s] 


    - Saved PDF: D:\PsyWiz\raw_data\article_3.pdf
    - Wrote metadata: D:\PsyWiz\raw_data\article_3_meta.json

Done. Downloaded & processed 2 papers to: D:\PsyWiz\raw_data
